In [19]:
import pandas as pd
import numpy as np
import tools


In [20]:
df_kalendarz = pd.read_parquet("dane/interim/fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet")


In [ ]:
df_kalendarz = pd.read_parquet("dane/interim/fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet")

df_sprzedaz = df_kalendarz[df_kalendarz['TypRuchu'] == 'sprzedaz'].copy()

towid_do_kalendarza = df_sprzedaz['TowId'].unique()
print(f"TowId z jakąkolwiek sprzedażą: {len(towid_do_kalendarza):,}")

data_min = df_kalendarz['Data'].min()
data_max = df_kalendarz['Data'].max()
kalendarz_dni = pd.date_range(start=data_min, end=data_max, freq='D')
print(f"Dni w zakresie: {len(kalendarz_dni):,}")

siatka = pd.MultiIndex.from_product(
    [towid_do_kalendarza, kalendarz_dni], names=['TowId', 'Data']
).to_frame(index=False)
print(f"Rozmiar siatki: {len(siatka):,}")

pelny_kalendarz = siatka.merge(df_sprzedaz, on=['TowId', 'Data'], how='left')
print(f"Rozmiar po scaleniu: {len(pelny_kalendarz):,}")

kolumny_towid = ['NazwaTow', 'EAN', 'AsId', 'Producent', 'NazwaAsort',
                  'NazwaTowCleanName', 'NazwaAsortCleanName', 'JestMartwy']

atrybuty_towid = df_kalendarz[['TowId'] + kolumny_towid].drop_duplicates(subset='TowId')

for kol in kolumny_towid:
    mapa = atrybuty_towid.set_index('TowId')[kol]
    pelny_kalendarz[kol] = pelny_kalendarz[kol].fillna(pelny_kalendarz['TowId'].map(mapa))

pelny_kalendarz['DokId'] = pelny_kalendarz['DokId'].fillna(-1).astype(int)


# Zakres życia per TowId (z bufor 14 dni,)
daty_sku = df_sprzedaz.groupby('TowId')['Data'].agg(DataStart='min', DataKoniec='max').reset_index()
daty_sku['DataKoniec'] = (daty_sku['DataKoniec'] + pd.Timedelta(days=14)).clip(upper=data_max)

# Dołączamy do pelny_kalendarz i filtrujemy
pelny_kalendarz = pelny_kalendarz.merge(daty_sku, on='TowId', how='left')

przed = len(pelny_kalendarz)
pelny_kalendarz = pelny_kalendarz[
    (pelny_kalendarz['Data'] >= pelny_kalendarz['DataStart']) &
    (pelny_kalendarz['Data'] <= pelny_kalendarz['DataKoniec'])
].copy()

print(f"Przed przycięciem: {przed:,} wierszy")
print(f"Po przycięciu:     {len(pelny_kalendarz):,} wierszy")

# Opcjonalnie: usuń pomocnicze kolumny DataStart/DataKoniec, jeśli nie chcesz ich w finalnym pliku
pelny_kalendarz = pelny_kalendarz.drop(columns=['DataStart', 'DataKoniec'])




print(f"\nPuste rekordy (DokId=-1): {(pelny_kalendarz['DokId']==-1).sum():,}")
print(f"Realne rekordy sprzedaży: {(pelny_kalendarz['DokId']!=-1).sum():,}")


TowId z jakąkolwiek sprzedażą: 12,481
Dni w zakresie: 1,127
Rozmiar siatki: 14,066,087
Rozmiar po scaleniu: 15,810,161
Przed przycięciem: 15,810,161 wierszy
Po przycięciu:     8,802,515 wierszy

Puste rekordy (DokId=-1): 5,428,540
Realne rekordy sprzedaży: 3,373,975


In [23]:
# Kontrola: żadna transakcja nie mogła zginąć ani się zduplikować przy scalaniu
print(f"Wiersze w df_sprzedaz (surowe): {len(df_sprzedaz):,}")
print(f"Realne rekordy w pelny_kalendarz: {(pelny_kalendarz['DokId']!=-1).sum():,}")
print(f"Zgodność: {(pelny_kalendarz['DokId']!=-1).sum() == len(df_sprzedaz)}")

print(f"\nRozmiar: {pelny_kalendarz.shape}")
print(f"Pamięć: {pelny_kalendarz.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nKolumny: {pelny_kalendarz.columns.tolist()}")


Wiersze w df_sprzedaz (surowe): 3,373,975
Realne rekordy w pelny_kalendarz: 3,373,975
Zgodność: True

Rozmiar: (8802515, 33)
Pamięć: 3960.5 MB

Kolumny: ['TowId', 'Data', 'DokId', 'Kolejnosc', 'NrPozycji', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPoRab', 'Wartosc', 'KolejnyWDniu', 'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty', 'Zaplacono', 'AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'AktywnyTow', 'NazwaAsort', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik', 'TypRuchu', 'CzyNiechciane', 'NazwaTowCleanName', 'NazwaAsortCleanName', 'JestMartwy']


In [24]:
pelny_kalendarz.to_parquet(
    "dane/interim/kalendarz_pelny_towid.parquet",
    compression='zstd',
    index=False
)


In [25]:
# Suma kontrolna
nazwa_pliku = "kalendarz_pelny_towid.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")


Mój hash (posortowane):   kalendarz_pelny_towid.parquet   cc4abda6396e405db53c1bc8e4e6dac56d1f1211d093111472093b324333771d


In [ ]:
# hash pliku kalendarz_pelny_towid.parquet:  cc4abda6396e405db53c1bc8e4e6dac56d1f1211d093111472093b324333771d

In [27]:
daty_sku

,TowId,DataStart,DataKoniec
0,15,2023-02-04,2025-12-15
1,49,2023-03-01,2026-01-31
2,53,2023-01-19,2026-01-31
3,55,2023-01-20,2025-12-11
4,56,2023-01-31,2026-01-27
...,...,...,...
12476,81356,2025-10-24,2026-01-31
12477,81357,2026-01-12,2026-01-31
12478,81358,2025-12-03,2026-01-31
12479,81359,2025-12-19,2026-01-31
